In [1]:
# Cell 1 - Configurar rutas
lakehouses = mssparkutils.lakehouse.list()
for lh in lakehouses:
    if 'Silver' in lh['displayName']:
        SILVER_ID = lh['id']
        WORKSPACE_ID = lh['workspaceId']
    if 'Bronze' in lh['displayName']:
        BRONZE_ID = lh['id']

BRONZE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"
SILVER_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"

print(f"Bronze: {BRONZE_ID}")
print(f"Silver: {SILVER_ID}")
print("Rutas configuradas OK")

StatementMeta(, 6e9718f3-da83-48e3-b2c1-f0419511afe0, 3, Finished, Available, Finished, False)

Bronze: f239c4cb-d96b-476a-8174-e8a857dbb690
Silver: 7238c65b-dcce-476a-9311-134ae8788a34
Rutas configuradas OK


In [2]:
# Cell 2 - Funciones de limpieza
from pyspark.sql.functions import col, trim, when, to_date, to_timestamp
from pyspark.sql.types import StringType, DateType, TimestampType, DecimalType, IntegerType, LongType, DoubleType
import re

def clean_table(df, table_name):
    """Aplica limpieza estándar a cualquier tabla"""
    
    report = {"table": table_name, "original_rows": df.count(), "issues": []}
    
    # 1 - Trim en columnas string y vacíos a null
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, 
                when(trim(col(field.name)) == "", None)
                .otherwise(trim(col(field.name)))
            )
    
    # 2 - Estandarizar nombres de columnas a snake_case
    def to_snake_case(name):
        s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
        return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    
    new_columns = [to_snake_case(c) for c in df.columns]
    for old, new in zip(df.columns, new_columns):
        if old != new:
            df = df.withColumnRenamed(old, new)
    
    # 3 - Detectar duplicados en primera columna (asumida como PK)
    pk_col = new_columns[0]
    total = df.count()
    distinct = df.select(pk_col).distinct().count()
    if total != distinct:
        report["issues"].append(f"Duplicados en {pk_col}: {total - distinct}")
    
    # 4 - Contar nulls en todas las columnas
    null_counts = {}
    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            null_counts[c] = null_count
    if null_counts:
        report["issues"].append(f"Nulls encontrados: {null_counts}")
    
    report["final_rows"] = df.count()
    report["columns"] = len(df.columns)
    
    return df, report

print("Funciones de limpieza definidas OK")

StatementMeta(, 6e9718f3-da83-48e3-b2c1-f0419511afe0, 4, Finished, Available, Finished, False)

Funciones de limpieza definidas OK


In [3]:
# Cell 3 - Procesar todas las tablas Bronze a Silver
tables = spark.catalog.listTables()
bronze_tables = [t.name for t in tables if t.name.startswith("bronze_")]

print(f"Tablas Bronze encontradas: {len(bronze_tables)}")

all_reports = []
errors = []

for bronze_table in bronze_tables:
    try:
        # Leer desde Bronze
        df = spark.table(bronze_table)
        
        # Limpiar
        df_clean, report = clean_table(df, bronze_table)
        
        # Nombre de tabla Silver
        silver_table = bronze_table.replace("bronze_", "silver_")
        
        # Guardar en Silver
        df_clean.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(silver_table)
        
        all_reports.append(report)
        status = "⚠️" if report["issues"] else "✅"
        print(f"{status} {silver_table} ({report['final_rows']:,} filas, {report['columns']} cols)")
        
    except Exception as e:
        errors.append(f"ERROR: {bronze_table} -> {str(e)[:100]}")
        print(f"❌ {bronze_table} -> ERROR")

print(f"\nTotal procesadas: {len(all_reports)}")
print(f"Total errores: {len(errors)}")

StatementMeta(, 6e9718f3-da83-48e3-b2c1-f0419511afe0, 5, Finished, Available, Finished, False)

Tablas Bronze encontradas: 65
✅ silver_humanresources_department (16 filas, 5 cols)
⚠️ silver_humanresources_employeedepartmenthistory (296 filas, 7 cols)
⚠️ silver_humanresources_employeepayhistory (316 filas, 6 cols)
⚠️ silver_humanresources_jobcandidate (13 filas, 5 cols)
✅ silver_person_addresstype (6 filas, 5 cols)
✅ silver_person_businessentity (20,777 filas, 4 cols)
⚠️ silver_person_businessentityaddress (19,614 filas, 6 cols)
⚠️ silver_person_businessentitycontact (909 filas, 6 cols)
✅ silver_person_contacttype (20 filas, 4 cols)
✅ silver_person_countryregion (238 filas, 4 cols)
✅ silver_person_emailaddress (19,972 filas, 6 cols)
✅ silver_person_password (19,972 filas, 6 cols)
⚠️ silver_person_person (19,972 filas, 14 cols)
✅ silver_person_personphone (19,972 filas, 5 cols)
✅ silver_person_phonenumbertype (3 filas, 4 cols)
✅ silver_person_stateprovince (181 filas, 9 cols)
⚠️ silver_production_billofmaterials (2,679 filas, 10 cols)
⚠️ silver_production_culture (8 filas, 4 cols)


In [4]:
# Cell 4 - Ver detalle de issues
print("Tablas con issues:\n")
for report in all_reports:
    if report["issues"]:
        print(f"📋 {report['table']}")
        for issue in report["issues"]:
            print(f"   {issue}")
        print()

StatementMeta(, 6e9718f3-da83-48e3-b2c1-f0419511afe0, 6, Finished, Available, Finished, False)

Tablas con issues:

📋 bronze_humanresources_employeedepartmenthistory
   Duplicados en business_entity_id: 6
   Nulls encontrados: {'end_date': 290}

📋 bronze_humanresources_employeepayhistory
   Duplicados en business_entity_id: 26

📋 bronze_humanresources_jobcandidate
   Nulls encontrados: {'business_entity_id': 11}

📋 bronze_person_businessentityaddress
   Duplicados en business_entity_id: 35

📋 bronze_person_businessentitycontact
   Duplicados en business_entity_id: 104

📋 bronze_person_person
   Nulls encontrados: {'title': 18963, 'middle_name': 8499, 'suffix': 19919, 'additional_contact_info': 19962}

📋 bronze_production_billofmaterials
   Nulls encontrados: {'product_assembly_id': 103, 'end_date': 2480}

📋 bronze_production_culture
   Nulls encontrados: {'culture_id': 1}

📋 bronze_production_product
   Nulls encontrados: {'color': 248, 'size': 293, 'size_unit_measure_code': 328, 'weight_unit_measure_code': 299, 'weight': 299, 'product_line': 226, 'class': 257, 'style': 293, 'pro